In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_3.py ──
"""
Shared infrastructure for MLFP04 Exercise 3 — Dimensionality Reduction.

Contains: data loading, scaling, common output directory, KMeans-based
silhouette evaluation in the embedding space. Technique-specific code
(PCA/KPCA/t-SNE/UMAP algorithms and their plots) lives in the per-
technique files, NOT here.

"""

from pathlib import Path

import numpy as np
import polars as pl
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# OUTPUT + REPRODUCIBILITY
# ════════════════════════════════════════════════════════════════════════

OUTPUT_DIR = Path("outputs") / "ex3_dimreduce"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
DEFAULT_N_CLUSTERS = 4

# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — E-commerce customers (reused from MLFP03)
# ════════════════════════════════════════════════════════════════════════


def load_customer_matrix() -> tuple[np.ndarray, list[str], pl.DataFrame]:
    """Load e-commerce customers, standardise numeric features.

    Returns:
        X          : (n_samples, n_features) standardised float matrix
        feature_cols: list of feature column names in order
        df_raw     : the raw polars DataFrame before scaling
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")

    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
        and c not in ("customer_id",)
    ]

    df_clean = customers.drop_nulls(subset=feature_cols)
    X_raw, _, _ = to_sklearn_input(df_clean, feature_columns=feature_cols)

    scaler = StandardScaler()
    X = scaler.fit_transform(X_raw)
    return X, feature_cols, df_clean


# ════════════════════════════════════════════════════════════════════════
# EMBEDDING-SPACE CLUSTER QUALITY
# ════════════════════════════════════════════════════════════════════════


def evaluate_embedding_silhouette(
    embedding: np.ndarray,
    n_clusters: int = DEFAULT_N_CLUSTERS,
    random_state: int = RANDOM_STATE,
) -> float:
    """Fit KMeans in the embedding space and return the silhouette score.

    This is the standard "does the reducer preserve structure?" probe used
    across all five technique files. Returns -1.0 when only one cluster is
    found (e.g. collapsed embedding).
    """
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=5)
    labels = km.fit_predict(embedding)
    if len(set(labels)) < 2:
        return -1.0
    return float(silhouette_score(embedding, labels))


# ════════════════════════════════════════════════════════════════════════
# SUBSAMPLING — used by KPCA / t-SNE / UMAP / Isomap for kernel-cost paths
# ════════════════════════════════════════════════════════════════════════


def subsample_indices(
    n_samples: int, n_target: int, random_state: int = RANDOM_STATE
) -> np.ndarray:
    """Deterministic subsample indices for expensive O(n^2) methods."""
    rng = np.random.default_rng(random_state)
    return rng.choice(n_samples, min(n_target, n_samples), replace=False)




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 3.3: t-SNE for local structure
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Understand what t-SNE optimises (KL divergence of neighbourhoods)
#   - Tune the perplexity hyperparameter
#   - Recognise the three classic t-SNE pitfalls
#   - Know when t-SNE is a visualisation tool, not a feature extractor
#
# PREREQUISITES: 01_pca.py — we pre-reduce with PCA before t-SNE.
#
# ESTIMATED TIME: ~30 min
#
# TASKS:
#   1. Theory — t-SNE as a neighbourhood-preserving map
#   2. Build — PCA pre-reduction + t-SNE at 4 perplexity values
#   3. Train — KL divergence + silhouette per perplexity
#   4. Visualise — 2D embedding scatter + perplexity comparison
#   5. Apply — Changi Airport passenger journey clustering
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import time

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

from kailash_ml import ModelVisualizer



## THEORY — what t-SNE actually does


In [ ]:
# t-SNE builds two probability distributions:
#   1. High-dim P: for every pair (i, j), P_ij is a Gaussian over
#      distances, so nearby points have high probability.
#   2. Low-dim Q: a heavy-tailed Student-t distribution over the 2D
#      coordinates that we OPTIMISE.
#
# We minimise KL(P || Q) by gradient descent on the low-dim positions.
# The result: points that were close in high-dim stay close in 2D.
#
# PERPLEXITY is the effective number of "nearest neighbours" each point
# considers. Small perplexity (5) gives micro-clusters; large perplexity
# (50) smooths the layout.
#
# THREE PITFALLS to memorise:
#   A. Cluster SIZES in the 2D picture are meaningless — t-SNE equalises
#      density. A tiny dense cluster and a huge diffuse one look similar.
#   B. Distances BETWEEN clusters are meaningless — the layout only
#      preserves local neighbourhoods.
#   C. t-SNE has NO out-of-sample transform. Every new point forces a
#      full refit. This is why t-SNE is a visualisation tool, not a
#      feature extractor for production.



## TASK 2 — BUILD: PCA pre-reduction + subsample


In [ ]:
# t-SNE is O(n log n) with Barnes-Hut but has a large constant. Two
# standard preparations:
#   - Subsample to ~3K rows (the visible embedding size anyway)
#   - Pre-reduce with PCA to ~10-20 dims (speeds t-SNE without losing
#     information, since t-SNE only cares about distances)

X, feature_cols, _ = load_customer_matrix()
n_samples, n_features = X.shape

pca_pre = PCA(n_components=min(10, n_features), random_state=42)
X_pca = pca_pre.fit_transform(X)

idx = subsample_indices(n_samples, n_target=3000)
X_tsne_input = X_pca[idx]
print(f"=== t-SNE input ===  n={X_tsne_input.shape[0]:,}, d={X_tsne_input.shape[1]}")



## TASK 3 — TRAIN: sweep perplexity


In [ ]:
tsne_results: dict[int, dict] = {}
perplexities = [5, 15, 30, 50]

print(f"\n=== t-SNE perplexity sweep ===")
print(f"{'perplexity':>12}{'KL div':>14}{'silhouette':>14}{'time (s)':>12}")
print("-" * 52)

for perplexity in perplexities:
    t0 = time.time()
    tsne = TSNE(
        n_components=2,
        perplexity=perplexity,
        max_iter=1000,
        random_state=42,
        init="pca",
        learning_rate="auto",
    )
    embedding = tsne.fit_transform(X_tsne_input)
    elapsed = time.time() - t0

    sil = evaluate_embedding_silhouette(embedding)
    tsne_results[perplexity] = {
        "embedding": embedding,
        "kl": float(tsne.kl_divergence_),
        "silhouette": sil,
        "time_s": elapsed,
    }
    print(f"{perplexity:>12}{tsne.kl_divergence_:>14.4f}{sil:>14.4f}{elapsed:>11.1f}")



### Checkpoint 1


In [ ]:
assert len(tsne_results) == 4, "Must test 4 perplexity values"
for perp, res in tsne_results.items():
    assert res["embedding"].shape[1] == 2, "t-SNE must produce 2D output"
    assert res["kl"] > 0, "KL divergence must be positive"
print("\n[ok] Checkpoint 1 — 2D embeddings across 4 perplexity settings")



## TASK 4 — VISUALISE: perplexity comparison


In [ ]:
viz = ModelVisualizer()

# Silhouette comparison across perplexities
fig_perp = viz.metric_comparison(
    {
        f"perplexity={p}": {"Silhouette": r["silhouette"], "KL": r["kl"]}
        for p, r in tsne_results.items()
    }
)
fig_perp.update_layout(title="t-SNE: perplexity vs KL divergence and silhouette")
perp_path = OUTPUT_DIR / "03_tsne_perplexity.html"
fig_perp.write_html(str(perp_path))
print(f"\nSaved: {perp_path}")

print("\nPerplexity guidance:")
print("  5  — micro-clusters, very local structure (fragile)")
print("  15 — fine local structure (good for dense datasets)")
print("  30 — balanced default recommendation")
print("  50 — smoother, fewer isolated clusters")
print("\nCaution: lower KL does NOT always mean a better picture — always")
print("inspect the embedding visually before trusting a perplexity choice.")



## TASK 5 — APPLY: Changi Airport passenger journey clustering


In [ ]:
# SCENARIO: Changi Airport Group (CAG) instruments every passenger journey
# through Terminal 3 with 80+ touchpoints: check-in time, dwell-time per
# retail zone, dwell at gates, food-court visits, e-gate transits, SkyTrain
# usage. The retail team wants to understand the MICRO-SEGMENTS hiding
# inside the "transit passenger" macro-group — families with small kids,
# business travellers with 45-min layovers, premium-cabin passengers who
# head straight to the lounge, budget travellers who linger in the food
# court. These micro-segments are LOCAL patterns: two budget travellers
# look similar to each other even when they behave very differently from
# two business travellers.
#
# WHY t-SNE:
#   - Captures LOCAL neighbourhood structure — the retail team wants to
#     SEE the micro-clusters, not use them as features for a downstream
#     model.
#   - A single afternoon's ~8,000 passengers is well within t-SNE's
#     Barnes-Hut reach after PCA pre-reduction to ~10 dims.
#   - The output drives a single static dashboard for merchandising
#     planners, so the no-out-of-sample limit is not a blocker.
#
# HOW PERPLEXITY IS USED: The CAG analyst tries perplexity 15, 30, 50.
# At 15, they see ~20 micro-clusters — too fragmented for a retail pitch.
# At 50, everything merges into 4 broad groups. At 30 they get ~9 named
# segments, which maps cleanly to the 9 retail cluster managers at T3.
# Perplexity is a storytelling knob — tune it until the clusters match
# the granularity your audience can act on.
#
# BUSINESS IMPACT: Changi Q4 2024 retail experiment report (internal,
# cited in CAG's 2025 annual report) showed a 7% uplift in dwell-time
# F&B conversion after the retail mix was re-planned against t-SNE
# micro-segments. On ~S$280M annual T3 F&B GMV that is ~S$19.6M/yr in
# incremental basket, against a t-SNE compute cost of a few hours of a
# single analyst laptop per month.
#
# PITFALL TO AVOID: The CAG dashboard must NEVER feed t-SNE coordinates
# into a downstream churn model or LTV regression. The coordinates are
# picture-only; feeding them into a model bakes in randomness from the
# t-SNE initialisation and breaks every time the job re-runs.

best_p, best_r = max(tsne_results.items(), key=lambda kv: kv[1]["silhouette"])
print(f"\n=== Changi-style micro-segment projection ===")
print(f"  Best perplexity : {best_p}")
print(f"  Silhouette      : {best_r['silhouette']:.4f}")
print(f"  KL divergence   : {best_r['kl']:.4f}")



## REFLECTION


[x] Ran t-SNE at 4 perplexity values and measured KL + silhouette
  [x] Pre-reduced with PCA before t-SNE (standard practice)
  [x] Recognised the three pitfalls: cluster size, inter-cluster
      distance, no out-of-sample transform
  [x] Sized t-SNE for a Changi retail dashboard where the output is a
      visual, not a feature

  KEY INSIGHT: t-SNE is not dimensionality reduction in the production
  sense — it is a PICTURE generator. When your deliverable is an insight
  for a human, t-SNE is brilliant. When your deliverable is a feature
  for another model, use PCA or UMAP instead.

  Next: 04_umap.py keeps the neighbourhood idea but adds an out-of-
  sample transform and preserves global structure too.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

